In [1]:
# ==========================================
# Inventory Optimization & Stock-out Prevention
# ==========================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve
)

import joblib

print("All libraries imported successfully.")

All libraries imported successfully.


In [2]:
# Load Dataset
df = pd.read_csv("feature_engineered_data.csv")

# Basic Information
print("Dataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nFirst 5 Rows:")
display(df.head())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDataset Info:")
df.info()

Dataset Shape: (779425, 19)

Column Names:
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'Revenue', 'Year', 'Month', 'MonthName', 'Day', 'DayOfWeek', 'Hour', 'Quarter', 'IsWeekend', 'BasketSize', 'OrderValue']

First 5 Rows:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue,Year,Month,MonthName,Day,DayOfWeek,Hour,Quarter,IsWeekend,BasketSize,OrderValue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4,2009,12,December,1,Tuesday,7,4,0,166,505.3
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,2009,12,December,1,Tuesday,7,4,0,166,505.3
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,2009,12,December,1,Tuesday,7,4,0,166,505.3
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8,2009,12,December,1,Tuesday,7,4,0,166,505.3
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0,2009,12,December,1,Tuesday,7,4,0,166,505.3



Missing Values:
Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
Revenue        0
Year           0
Month          0
MonthName      0
Day            0
DayOfWeek      0
Hour           0
Quarter        0
IsWeekend      0
BasketSize     0
OrderValue     0
dtype: int64

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 779425 entries, 0 to 779424
Data columns (total 19 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Invoice      779425 non-null  int64  
 1   StockCode    779425 non-null  object 
 2   Description  779425 non-null  object 
 3   Quantity     779425 non-null  int64  
 4   InvoiceDate  779425 non-null  object 
 5   Price        779425 non-null  float64
 6   Customer ID  779425 non-null  float64
 7   Country      779425 non-null  object 
 8   Revenue      779425 non-null  float64
 9   Year         779425 non-null  int64  
 10  

In [3]:
# ==========================================
# Inventory Optimization & Stock-out Label
# ==========================================

# Low Stock Threshold
threshold = 5

# Create StockOut Label
df["StockOut"] = np.where(df["Quantity"] <= threshold, 1, 0)

print(df["StockOut"].value_counts())

print("\nStockOut Percentage:")
print(df["StockOut"].value_counts(normalize=True) * 100)

StockOut
0    390914
1    388511
Name: count, dtype: int64

StockOut Percentage:
StockOut
0    50.154152
1    49.845848
Name: proportion, dtype: float64


In [4]:
# ==========================================
# Encode StockCode
# ==========================================

encoder = LabelEncoder()

df["StockCode_Encoded"] = encoder.fit_transform(df["StockCode"])

print("StockCode encoded successfully.")

print(df[["StockCode", "StockCode_Encoded"]].head())

StockCode encoded successfully.
  StockCode  StockCode_Encoded
0     85048               4009
1    79323P               3326
2    79323W               3328
3     22041               1253
4     21232                618


In [9]:
# ==========================================
# Select Features and Target
# ==========================================

features = [
    "StockCode_Encoded",
    "Revenue",
    "OrderValue",
    "BasketSize"
]

X = df[features]

y = df["StockOut"]

print("Feature Shape:", X.shape)
print("Target Shape:", y.shape)

Feature Shape: (779425, 4)
Target Shape: (779425,)


In [10]:
# ==========================================
# Train Test Split
# ==========================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Shape :", X_train.shape)
print("Testing Shape  :", X_test.shape)

Training Shape : (623540, 4)
Testing Shape  : (155885, 4)


In [11]:
# ==========================================
# Train Inventory Optimization Model
# ==========================================

from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

print("Inventory Optimization Model Trained Successfully.")

Inventory Optimization Model Trained Successfully.


In [12]:
# Prediction

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

In [13]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

print("Accuracy :", accuracy_score(y_test, y_pred))
print()

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print()

print("ROC-AUC Score :", roc_auc_score(y_test, y_prob))

Accuracy : 0.9821470956153575

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.98      0.98     78183
           1       0.98      0.98      0.98     77702

    accuracy                           0.98    155885
   macro avg       0.98      0.98      0.98    155885
weighted avg       0.98      0.98      0.98    155885

Confusion Matrix:
[[76752  1431]
 [ 1352 76350]]

ROC-AUC Score : 0.9983730380385376


In [14]:
import pandas as pd

importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

importance = importance.sort_values(by="Importance", ascending=False)

print(importance)

             Feature  Importance
1            Revenue    0.642744
0  StockCode_Encoded    0.212630
3         BasketSize    0.078294
2         OrderValue    0.066331


In [15]:
import joblib

joblib.dump(model, "inventory_model.pkl")

print("Inventory model saved successfully.")

Inventory model saved successfully.


In [16]:
joblib.dump(encoder, "stockcode_encoder.pkl")

print("StockCode encoder saved successfully.")

StockCode encoder saved successfully.


In [17]:
result = X_test.copy()

result["Actual"] = y_test.values
result["Predicted"] = y_pred

result.to_csv("inventory_predictions.csv", index=False)

print("Inventory prediction file saved successfully.")

Inventory prediction file saved successfully.


In [18]:
print("="*50)
print("Inventory Optimization & Stock-out Prevention Completed")
print("="*50)
print("Model : Random Forest Classifier")
print("Features Used :")
print("- StockCode")
print("- Revenue")
print("- OrderValue")
print("- BasketSize")
print("\nFiles Generated:")
print("- inventory_model.pkl")
print("- stockcode_encoder.pkl")
print("- inventory_predictions.csv")
print("="*50)

Inventory Optimization & Stock-out Prevention Completed
Model : Random Forest Classifier
Features Used :
- StockCode
- Revenue
- OrderValue
- BasketSize

Files Generated:
- inventory_model.pkl
- stockcode_encoder.pkl
- inventory_predictions.csv


In [19]:
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

importance = importance.sort_values(by="Importance", ascending=False)

print(importance)

             Feature  Importance
1            Revenue    0.642744
0  StockCode_Encoded    0.212630
3         BasketSize    0.078294
2         OrderValue    0.066331


In [20]:
import joblib

joblib.dump(model, "inventory_model.pkl")

print("Inventory model saved successfully.")

Inventory model saved successfully.


In [21]:
joblib.dump(encoder, "stockcode_encoder.pkl")

print("StockCode encoder saved successfully.")

StockCode encoder saved successfully.


In [22]:
result = X_test.copy()

result["Actual"] = y_test.values
result["Predicted"] = y_pred

result.to_csv("inventory_predictions.csv", index=False)

print("Inventory prediction file saved successfully.")

Inventory prediction file saved successfully.


In [23]:
print("="*60)
print("Inventory Optimization & Stock-out Prevention Completed")
print("="*60)

print("Model Used : Random Forest Classifier")

print("\nFeatures Used:")
print("- StockCode")
print("- Revenue")
print("- OrderValue")
print("- BasketSize")

print("\nGenerated Files:")
print("- inventory_model.pkl")
print("- stockcode_encoder.pkl")
print("- inventory_predictions.csv")

print("\nPerformance:")
print("Accuracy :", accuracy_score(y_test, y_pred))
print("ROC-AUC :", roc_auc_score(y_test, y_prob))

print("="*60)

Inventory Optimization & Stock-out Prevention Completed
Model Used : Random Forest Classifier

Features Used:
- StockCode
- Revenue
- OrderValue
- BasketSize

Generated Files:
- inventory_model.pkl
- stockcode_encoder.pkl
- inventory_predictions.csv

Performance:
Accuracy : 0.9821470956153575
ROC-AUC : 0.9983730380385376
